In [1]:
import os, re
import numpy as np
import pandas as pd
from rapidfuzz import fuzz

schools = pd.read_csv("../data/processed/schools_with_athletics.csv")
sevp_all = pd.read_csv("../data/raw/sevp/sevp_certified_schools.csv")
sevp = sevp_all[sevp_all["f_visa"] == "Y"].copy()   # only F-1 matters for degree students
# Keep v1 results to compare what changed
old_path = "../data/processed/schools_with_sevp.csv"
old = pd.read_csv(old_path)[["unit_id", "sevp_certified"]] if os.path.exists(old_path) else None
print("Schools:", schools.shape, "| SEVP F-1 rows:", sevp.shape)

Schools: (3147, 44) | SEVP F-1 rows: (13267, 8)


In [2]:
ABBREV = {
    r"\bcoll\b": "college",
    r"\buniv\b": "university",
    r"\bcc\b": "community college",
    r"\bcomm\b": "community",
    r"\bctr\b": "center",
    r"\bmt\b": "mount",
    r"\bco\b": "county",
    r"\bsaint\b": "st",
}
# Manual crosswalk for true aliases no rule can catch (add to this as you find them)
ALIASES = {
    "university at buffalo": "state university of new york at buffalo",
}
STOP = {"of", "and", "at", "in", "for"}
GENERIC = {"college", "university", "community", "campus", "school", "institute",
           "center", "state", "technical", "county", "st"}

def norm(s):
    s = str(s).lower().replace("&", " and ")
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    for pat, rep in ABBREV.items():
        s = re.sub(pat, rep, s)
    s = re.sub(r"\b(the|inc|llc)\b", " ", s)
    s = re.sub(r"\bmain campus\b", " ", s)
    s = re.sub(r"^\s*(cuny|suny)\s+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def parent_name(s):
    # Text before the LAST hyphen or slash: "X University-Some Campus" -> "X University"
    s = str(s)
    idx = max(s.rfind("-"), s.rfind("/"))
    return norm(s[:idx]) if idx > 0 else None

def tokens(s):
    return set(str(s).split()) - STOP

schools["name_n"] = schools["name"].apply(norm).replace(ALIASES)
schools["parent_n"] = schools["name"].apply(parent_name)
schools["city_n"] = schools["city"].apply(norm)
sevp["school_n"] = sevp["school_name"].apply(norm)
sevp["campus_n"] = sevp["campus_name"].apply(norm)
sevp["school_p"] = sevp["school_name"].apply(parent_name)   # "Highland Community College-Kansas" -> "highland community college"
sevp["campus_p"] = sevp["campus_name"].apply(parent_name)
sevp["city_n"] = sevp["city"].apply(norm)

In [3]:
sevp_by_state = {st: g for st, g in sevp.groupby("state")}   # blocking: same state only

def link(row):
    cand = sevp_by_state.get(row["state"])
    if cand is None:
        return pd.Series([None, None, None, 0.0])
    out = lambda method, h, sc: pd.Series([method, h["school_name"], h["campus_name"], sc])

    # Stage 1: exact normalized name
    hit = cand[cand["school_n"].eq(row["name_n"]) | cand["campus_n"].eq(row["name_n"])]
    if len(hit):
        return out("exact", hit.iloc[0], 100.0)

    # Stage 2: parent names on EITHER side (our branch suffix, or SEVP's suffix)
    names = {row["name_n"]} | ({row["parent_n"]} if row["parent_n"] else set())
    hit = cand[cand["school_n"].isin(names) | cand["campus_n"].isin(names) |
               cand["school_p"].isin(names) | cand["campus_p"].isin(names)]
    if len(hit):
        return out("parent", hit.iloc[0], 100.0)

    # Stage 3: subset rule, same city only. Shorter name's words all appear in the longer name,
    # and the shorter name has at least one distinctive (non-generic) word.
    same_city = cand[cand["city_n"] == row["city_n"]]
    mine = tokens(row["name_n"])
    for _, h in same_city.iterrows():
        for other in (h["school_n"], h["campus_n"]):
            theirs = tokens(other)
            short, longer = (mine, theirs) if len(mine) <= len(theirs) else (theirs, mine)
            if len(short - GENERIC) >= 1 and short <= longer:
                return out("subset", h, 100.0)

    # Stage 4: fuzzy. Prefer same-city candidates (accept 85+); otherwise require 97+.
    def best_of(c):
        s = np.maximum(c["school_n"].apply(lambda x: fuzz.token_sort_ratio(row["name_n"], x)),
                       c["campus_n"].apply(lambda x: fuzz.token_sort_ratio(row["name_n"], x)))
        i = s.idxmax()
        return c.loc[i], float(s[i])
    if len(same_city):
        h, sc = best_of(same_city)
        if sc >= 85:
            return out("fuzzy", h, round(sc, 1))
    h, sc = best_of(cand)
    if sc >= 97:
        return out("fuzzy", h, round(sc, 1))
    return pd.Series([None, h["school_name"], h["campus_name"], round(sc, 1)])  # best guess, rejected

schools[["sevp_match", "sevp_school", "sevp_campus", "sevp_score"]] = schools.apply(link, axis=1)

# F-1 students can't enroll in fully online programs: online units don't inherit certification
online = schools["name"].str.contains(r"online|extended education|non-traditional|professional programs|distance",
                                      case=False, regex=True, na=False)
schools.loc[online & schools["sevp_match"].notna(), "sevp_match"] = "online_excluded"
schools["sevp_certified"] = schools["sevp_match"].isin(["exact", "parent", "subset", "fuzzy"])

print("Match method:")
print(schools["sevp_match"].value_counts(dropna=False).to_string())
print("\nCertified by school type:")
print(pd.crosstab(schools["school_type"], schools["sevp_certified"], margins=True))
cols = ["name", "state", "sevp_school", "sevp_campus", "sevp_score"]
print("\n15 random SUBSET matches (review):")
sm = schools[schools["sevp_match"] == "subset"]
print(sm.sample(min(15, len(sm)), random_state=1)[cols].to_string(index=False))
print("\nOnline units excluded:")
print(schools[schools["sevp_match"] == "online_excluded"][["name", "state"]].to_string(index=False))

Match method:
sevp_match
exact              2277
None                545
parent              162
subset              132
fuzzy                17
online_excluded      11
NaN                   3

Certified by school type:
sevp_certified  False  True   All
school_type                      
2-year            403   947  1350
4-year            156  1641  1797
All               559  2588  3147

15 random SUBSET matches (review):
                                                                       name state                                         sevp_school                             sevp_campus  sevp_score
                                         Kent State University at Ashtabula    OH                               Kent State University                               Ashtabula       100.0
                                              Maine College of Art & Design    ME                                Maine College of Art                    Maine College of Art       100.0
                

In [4]:
check = ["Duke University", "University at Buffalo", "CUNY LaGuardia Community College",
         "Butler Community College", "Highland Community College", "The New England Conservatory of Music",
         "Dordt University", "North Georgia Technical College", "Southeastern Community College",
         "Southcentral Kentucky Community and Technical College", "Mississippi Delta Community College",
         "Centra College"]
print("Check list:")
print(schools[schools["name"].isin(check)][
    ["name", "state", "city", "sevp_match", "sevp_school", "sevp_campus", "sevp_score"]
].to_string(index=False))

miss = schools[(~schools["sevp_certified"]) & (schools["pct_international"] >= 0.02)]
print("\nLikely linkage misses (2%+ international, not matched):", len(miss))
print(miss.nlargest(25, "pct_international")[
    ["name", "state", "pct_international", "sevp_school", "sevp_score"]
].round(3).to_string(index=False))

if old is not None:
    cmp = old.merge(schools[["unit_id", "name", "state", "sevp_certified", "sevp_match"]], on="unit_id", suffixes=("_prev", ""))
    gained = cmp[cmp["sevp_certified"] & ~cmp["sevp_certified_prev"]]
    lost = cmp[~cmp["sevp_certified"] & cmp["sevp_certified_prev"]]
    print(f"\nPrevious run -> this run: gained {len(gained)}, lost {len(lost)}")
    print("Lost:")
    print(lost[["name", "state", "sevp_match"]].to_string(index=False))

schools.to_csv("../data/processed/schools_with_sevp.csv", index=False)
print("\nSaved:", schools.shape)

Check list:
                                                 name state             city sevp_match                                      sevp_school                             sevp_campus  sevp_score
                      North Georgia Technical College    GA     Clarkesville       None                  South Georgia Technical College         South Georgia Technical College        93.5
                           Highland Community College    IL         Freeport      exact                       Highland Community College              Highland Community College       100.0
                                     Dordt University    IA     Sioux Center     subset                   Dordt University, Incorporated          Dordt University, Incorporated       100.0
                       Southeastern Community College    IA  West Burlington     parent Southeastern Community College - West Burlington Southeastern Community College - Keokuk       100.0
                             Butler Communi


Saved: (3147, 52)
